In [76]:
import pandas as pd

df = pd.read_csv('data/customer_shopping_behavior.csv')
df.head()
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [77]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [78]:
df['Review Rating'] = df.groupby ('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [79]:
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [80]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

In [81]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [82]:
#create a column age_group
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [83]:
df[['age', 'age_group']].head(10)  

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [84]:
# create column purchase_frequency_days
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

    

In [85]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)


,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [86]:
df[['discount_applied', 'promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [87]:
#Question: Do we need both of these columns, is one of them redundant? 
(df['discount_applied']== df['promo_code_used']).all()

np.True_

In [88]:
#2 columns contain exactly the same info, remove one of columns
df = df.drop('promo_code_used', axis=1)

In [89]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

In [90]:
# Save cleaned data
df.to_csv('data/customer_shopping_behavior_cleaned.csv', index=False)
print("✅ Cleaned data saved!")

✅ Cleaned data saved!


In [175]:
# Load cleaned data 
df = pd.read_csv('data/customer_shopping_behavior_cleaned.csv')
print(df.columns.tolist())

['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'shipping_type', 'discount_applied', 'previous_purchases', 'payment_method', 'frequency_of_purchases', 'age_group', 'purchase_frequency_days']


In [176]:
!pip install pyodbc sqlalchemy pandas


In [177]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [179]:
from sqlalchemy import create_engine
from sqlalchemy import text
import pandas as pd
import urllib

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DOBBY\\SQLEXPRESS;"
    "DATABASE=RetailDB;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

with engine.connect() as conn:
    result = conn.execute(text("SELECT DB_NAME() AS current_db"))
    print(result.fetchone())

('RetailDB',)


In [180]:
df = pd.read_csv('data/customer_shopping_behavior_cleaned.csv')
df.to_sql("customer", engine, if_exists="replace", index=False)
print("✅ Data loaded into RetailDB!")

✅ Data loaded into RetailDB!


In [181]:
pd.read_sql("SELECT TOP 3 * FROM customer", engine)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7


In [182]:
# Q1. What is the total revenue generated by male vs. female customers? (COMPARING REVENUE ACROSS DEMOGRAPHICS)

q1 = """
SELECT
    gender,
    COUNT (*) AS total_customers, 
    SUM (purchase_amount) as total_revenue,
    ROUND(AVG(purchase_amount), 2) AS avg_spend
FROM customer
GROUP BY gender
"""
pd.read_sql(q1, engine)

,gender,total_customers,total_revenue,avg_spend
0,Male,2652,157890,59
1,Female,1248,75191,60


In [183]:
#--Q2. Which customers used a discount but still spent more than the average purchase amount? 
q2 = """
SELECT 
    customer_id,
    purchase_amount,
    discount_applied
FROM customer
WHERE discount_applied = 'Yes'
AND purchase_amount > (SELECT AVG(purchase_amount) FROM customer)
ORDER BY purchase_amount DESC
"""
pd.read_sql(q2, engine)

,customer_id,purchase_amount,discount_applied
0,43,100,Yes
1,96,100,Yes
2,194,100,Yes
3,205,100,Yes
4,244,100,Yes
...,...,...,...
834,1333,60,Yes
835,1296,60,Yes
836,1247,60,Yes
837,1434,60,Yes


In [184]:
# Q3. Top 5 products with highest average review rating
q3 = """
SELECT TOP 10
    item_purchased,
    ROUND(AVG(review_rating), 2) AS avg_rating,
    COUNT(*) AS total_purchases
FROM customer
GROUP BY item_purchased
ORDER BY avg_rating DESC
"""
pd.read_sql(q3, engine)

,item_purchased,avg_rating,total_purchases
0,Gloves,3.86,140
1,Sandals,3.84,160
2,Boots,3.82,144
3,Hat,3.80,154
4,Handbag,3.78,153
5,Skirt,3.78,158
6,T-shirt,3.78,147
7,Jacket,3.76,163
8,Belt,3.76,161
9,Jewelry,3.76,171


In [185]:
# Q4. Compare the average Purchase Amounts between Standard and Express Shipping. 
q4 = """
SELECT 
    shipping_type,
    ROUND(AVG(purchase_amount), 2) AS avg_purchase
FROM customer
WHERE shipping_type IN ('Standard', 'Express')
GROUP BY shipping_type
"""
pd.read_sql(q4, engine)

,shipping_type,avg_purchase
0,Standard,58
1,Express,60


In [186]:
# Q5. Do subscribed customers spend more? Compare average spend and total revenue 
## between subcribers and non- subcribers
q5 = """
SELECT 
    subscription_status,
    COUNT(*) AS total_customers,
    ROUND(AVG(purchase_amount), 2) AS avg_spend,
    SUM(purchase_amount) AS total_revenue
FROM customer
GROUP BY subscription_status
ORDER BY total_revenue, avg_spend desc;
"""
pd.read_sql(q5, engine)

,subscription_status,total_customers,avg_spend,total_revenue
0,Yes,1053,59,62645
1,No,2847,59,170436


In [187]:
#Q6:Which 5 products have the highest percentage of purchases with discounts applied?
q6 = """
SELECT TOP 5
    item_purchased,
    CAST (
      ROUND(
       SUM (CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END) *100/ COUNT(*), 2) AS DECIMAL (10,2)) AS discount_percentage
FROM customer
GROUP BY item_purchased
ORDER BY discount_percentage DESC;
"""
pd.read_sql(q6, engine)

,item_purchased,discount_percentage
0,Hat,50.0
1,Coat,49.0
2,Sneakers,49.0
3,Sweater,48.0
4,Pants,47.0


In [152]:
# Q7. Segment customers into New, Returning, and Loyal based on their total 
q7 = """
WITH customer_type AS (
    SELECT customer_id, previous_purchases,
        CASE 
            WHEN previous_purchases = 1 THEN 'New'
            WHEN previous_purchases BETWEEN 2 AND 10 THEN 'Returning'
            ELSE 'Loyal'
        END AS customer_segment
     FROM customer
)

SELECT customer_segment, COUNT(*) as "number of customers"
from customer_type
group by customer_segment;
"""
pd.read_sql(q7, engine)

,customer_segment,number of customers
0,Returning,701
1,Loyal,3116
2,New,83


In [188]:
# Q8. What are the top 3 most purchased products within each category?
q8 = """
WITH product_counts AS (
    SELECT 
        category,
        item_purchased,
        COUNT(*) AS purchase_count
    FROM customer
    GROUP BY category, item_purchased
),

ranked_products AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY category 
            ORDER BY purchase_count DESC
        ) AS rank
    FROM product_counts
)

SELECT 
    category,
    item_purchased,
    purchase_count
FROM ranked_products
WHERE rank <= 3;
"""
pd.read_sql(q8, engine)

,category,item_purchased,purchase_count
0,Accessories,Jewelry,171
1,Accessories,Belt,161
2,Accessories,Sunglasses,161
3,Clothing,Blouse,171
4,Clothing,Pants,171
5,Clothing,Shirt,169
6,Footwear,Sandals,160
7,Footwear,Shoes,150
8,Footwear,Sneakers,145
9,Outerwear,Jacket,163


In [189]:
#Q9. Are customers who are repeat buyers (more than 5 previous purchases) also likely to subcribe?
q9 = """

SELECT subscription_status,
    COUNT(customer_id) AS repeat_buyers
From customer
WHERE previous_purchases > 5
GROUP BY subscription_status;
"""
pd.read_sql(q9, engine)


,subscription_status,repeat_buyers
0,Yes,958
1,No,2518


In [190]:
# Q10. Revenue contribution by age group
q10 = """
SELECT 
    age_group,
    COUNT(*) AS total_customers,
    SUM(purchase_amount) AS total_revenue,
    ROUND(AVG(purchase_amount), 2) AS avg_spend
FROM customer
GROUP BY age_group
ORDER BY age_group
"""
pd.read_sql(q10, engine)

,age_group,total_customers,total_revenue,avg_spend
0,Adult,942,55978,59
1,Middle-aged,986,59197,60
2,Senior,944,55763,59
3,Young Adult,1028,62143,60
